# 🇻🇳🇬🇧 Huấn Luyện Transformer Dịch Máy Anh - Việt (Bản Nâng Cấp Cao Cấp)
### Tận dụng tối đa dữ liệu mở: `Helsinki-NLP/opus-100` + `VietAI_MTet` + Từ điển đơn `Multilingual Core Vocabulary`
---
**Điểm đột phá của phiên bản này:**
1. **Tối đa hóa dữ liệu mở (Public & 100% Ungated):** Kết hợp câu đàm thoại đời sống (`OPUS-100`), văn phong học thuật chuẩn mực (`VietAI_MTet`) và bộ từ vựng đơn 16.800 từ gốc (`Multilingual-Core-Vocabulary`).
2. **Hỗ trợ Resume Training:** Cho phép nạp lại checkpoint cũ (`best_transformer_old.pt`) từ Google Drive để train tiếp nối (Epoch 9 -> 16) mà không phải học lại từ đầu.
3. **Kiến trúc Transformer Chuẩn:** 4 lớp Encoder / 4 lớp Decoder ($d_{model}=512, n_{heads}=8, d_{ff}=2048$), sửa triệt để lỗi FP16 overflow (`-1e4`).
4. **Tối ưu tốc độ:** Dynamic Padding, Batch Size 128 (hoặc 96), FP16 Mixed Precision với cú pháp mới `torch.amp`.

In [ ]:
# 1. Cài đặt thư viện cần thiết & Kết nối Google Drive
!pip install -q datasets tokenizers evaluate sacrebleu

import os
import sys
import math
import time
import json
import urllib.request
import csv
import io
from typing import List, Tuple, Union
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

# Kiểm tra GPU (Khuyên dùng: L4 hoặc T4)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'⭐ Thiết bị sử dụng: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

# Kết nối Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/Translator_PhoMT'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f'✅ Đã kết nối Google Drive! Checkpoint lưu tại: {SAVE_DIR}')
except Exception:
    SAVE_DIR = './models'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f'⚠️ Chạy chế độ cục bộ. Checkpoint lưu tại: {SAVE_DIR}')


In [ ]:
# 2. TẬN DỤNG BỘ DỮ LIỆU VÀNG PhoMT (VinAI) + CÂU CHỌN LỌC + DANH TỪ RIÊNG
import gc
import sqlite3
import urllib.request
import os
import zipfile
import torch
from datasets import load_dataset

# 1. Dọn sạch VRAM GPU
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'🧹 VRAM GPU khả dụng: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

# 2. Tự động tìm và giải nén PhoMT từ Google Drive nếu có
phomt_zip_drive = os.path.join(SAVE_DIR, 'PhoMT_detok.zip')
phomt_extracted_dir = '/content/PhoMT_data'

if os.path.exists(phomt_zip_drive):
    print(f'⭐ ĐÃ TÌM THẤY PhoMT TRÊN GOOGLE DRIVE: {phomt_zip_drive}')
    if not os.path.exists(phomt_extracted_dir):
        print('⏳ Đang giải nén PhoMT vào ổ đĩa tốc độ cao Colab...')
        with zipfile.ZipFile(phomt_zip_drive, 'r') as z:
            z.extractall(phomt_extracted_dir)
        print('✅ Giải nén PhoMT thành công!')
else:
    print(f'ℹ️ Chưa thấy file PhoMT_detok.zip trong {SAVE_DIR}.')
    print('   -> Hãy kéo file Translator/data/PhoMT_detok.zip (169 MB) thả vào thư mục Translator_PhoMT trên Google Drive!')

train_en, train_vi = [], []
dev_en, dev_vi = [], []
MAX_WORDS = 64

# 3. Đọc dữ liệu từ PhoMT (VinAI) - ƯU TIÊN HÀNG ĐẦU
phomt_train_en = os.path.join(phomt_extracted_dir, 'train', 'train.en')
phomt_train_vi = os.path.join(phomt_extracted_dir, 'train', 'train.vi')
phomt_dev_en = os.path.join(phomt_extracted_dir, 'dev', 'dev.en')
phomt_dev_vi = os.path.join(phomt_extracted_dir, 'dev', 'dev.vi')

if os.path.exists(phomt_train_en) and os.path.exists(phomt_train_vi):
    print('⏳ Đang nạp các câu văn bản chuẩn mực từ VinAI PhoMT...')
    with open(phomt_train_en, 'r', encoding='utf-8') as fe, open(phomt_train_vi, 'r', encoding='utf-8') as fv:
        for en_line, vi_line in zip(fe, fv):
            en, vi = en_line.strip(), vi_line.strip()
            if en and vi and 2 <= len(en.split()) <= MAX_WORDS and 2 <= len(vi.split()) <= MAX_WORDS:
                train_en.append(en)
                train_vi.append(vi)
                if len(train_en) >= 800000: break
    print(f'-> Đã nạp thành công {len(train_en):,} câu văn bản chuẩn ngữ pháp từ VinAI PhoMT!')

    with open(phomt_dev_en, 'r', encoding='utf-8') as fe, open(phomt_dev_vi, 'r', encoding='utf-8') as fv:
        for en_line, vi_line in zip(fe, fv):
            en, vi = en_line.strip(), vi_line.strip()
            if en and vi and 2 <= len(en.split()) <= MAX_WORDS and 2 <= len(vi.split()) <= MAX_WORDS:
                dev_en.append(en)
                dev_vi.append(vi)
                if len(dev_en) >= 4000: break
else:
    print('⏳ Dự phòng: Nạp từ OPUS-100 và VietAI MTet...')
    ds_opus = load_dataset('Helsinki-NLP/opus-100', 'en-vi')
    ds_mtet = load_dataset('wanhin/VietAI_MTet', split='train[:200000]')
    for item in ds_opus['train']:
        trans = item.get('translation', {})
        en, vi = trans.get('en', '').strip(), trans.get('vi', '').strip()
        if en and vi and 2 <= len(en.split()) <= MAX_WORDS and 2 <= len(vi.split()) <= MAX_WORDS:
            train_en.append(en)
            train_vi.append(vi)
            if len(train_en) >= 500000: break
    for item in ds_mtet:
        en, vi = item.get('en', '').strip(), item.get('vi', '').strip()
        if en and vi and 2 <= len(en.split()) <= MAX_WORDS and 2 <= len(vi.split()) <= MAX_WORDS:
            train_en.append(en)
            train_vi.append(vi)
    for item in ds_opus['validation']:
        trans = item.get('translation', {})
        en, vi = trans.get('en', '').strip(), trans.get('vi', '').strip()
        if en and vi and 2 <= len(en.split()) <= MAX_WORDS and 2 <= len(vi.split()) <= MAX_WORDS:
            dev_en.append(en)
            dev_vi.append(vi)
            if len(dev_en) >= 3500: break

# 4. Trộn thêm CSDL từ điển 104.829 từ để không bỏ sót từ đơn
db_url = 'https://raw.githubusercontent.com/skypediacode/english-vietnamese-dictionary/main/dictionary_en_vi.db'
db_path = 'dictionary_en_vi.db'
if not os.path.exists(db_path):
    urllib.request.urlretrieve(db_url, db_path)

conn = sqlite3.connect(db_path)
cur = conn.cursor()
cur.execute('SELECT w.word, d.definition FROM words w JOIN word_definitions wd ON w.id = wd.word_id JOIN definitions d ON wd.definition_id = d.id WHERE length(w.word) <= 30')
for word, defn in cur.fetchall():
    clean_def = defn.split(';')[0].split(',')[0].strip()
    if clean_def and len(clean_def) <= 35:
        train_en.append(word.strip().lower())
        train_vi.append(clean_def)
conn.close()

# 5. Dạy mô hình phân biệt Danh từ riêng (Apple, Turkey, Elon Musk,...)
entity_pairs = [
    ('Apple is a great company.', 'Apple là một công ty tuyệt vời.'),
    ('I want to eat an apple.', 'Tôi muốn ăn một quả táo.'),
    ('Turkey is a country.', 'Thổ Nhĩ Kỳ là một đất nước.'),
    ('We eat turkey on Thanksgiving.', 'Chúng ta ăn gà tây vào lễ Tạ Ơn.'),
    ('Elon Musk founded SpaceX.', 'Elon Musk thành lập SpaceX.'),
    ('Bill Gates visited Vietnam.', 'Bill Gates đã đến thăm Việt Nam.'),
    ('Steve Jobs was the CEO of Apple.', 'Steve Jobs từng là CEO của Apple.')
]
for en, vi in entity_pairs * 10:
    train_en.append(en)
    train_vi.append(vi)

print(f'\n🎉 TỔNG KẾT QUY MÔ HUẤN LUYỆN ĐẲNG CẤP:')
print(f'   - Tập Train: {len(train_en):,} câu (VinAI PhoMT + 160k từ điển + Danh từ riêng)')
print(f'   - Tập Dev:   {len(dev_en):,} câu')


In [ ]:
# 3. Huấn luyện hoặc nạp lại BPE Tokenizer (Vocab: 24.000)
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

SPECIAL_TOKENS = ['<pad>', '<sos>', '<eos>', '<unk>']
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3
VOCAB_SIZE = 24000

class TranslationTokenizer:
    def __init__(self, tokenizer_obj=None):
        if tokenizer_obj:
            self.tokenizer = tokenizer_obj
        else:
            self.tokenizer = Tokenizer(BPE(unk_token='<unk>'))
            self.tokenizer.pre_tokenizer = Whitespace()

    def train(self, texts, vocab_size=VOCAB_SIZE):
        trainer = BpeTrainer(vocab_size=vocab_size, min_frequency=2, special_tokens=SPECIAL_TOKENS)
        self.tokenizer.train_from_iterator(texts, trainer=trainer)

    def encode(self, text, add_special_tokens=True):
        ids = self.tokenizer.encode(text).ids
        return [SOS_IDX] + ids + [EOS_IDX] if add_special_tokens else ids

    def decode(self, token_ids):
        if isinstance(token_ids, torch.Tensor):
            token_ids = token_ids.detach().cpu().tolist()
        cleaned = []
        for tid in token_ids:
            if tid == EOS_IDX:
                break
            if tid not in (PAD_IDX, SOS_IDX, UNK_IDX):
                cleaned.append(tid)
        return self.tokenizer.decode(cleaned)

en_tok_path = os.path.join(SAVE_DIR, 'tokenizer_en.json')
vi_tok_path = os.path.join(SAVE_DIR, 'tokenizer_vi.json')

if os.path.exists(en_tok_path) and os.path.exists(vi_tok_path):
    print('⭐ Đã tìm thấy Tokenizer sẵn có trong Drive. Đang nạp lại...')
    tok_en = TranslationTokenizer(Tokenizer.from_file(en_tok_path))
    tok_vi = TranslationTokenizer(Tokenizer.from_file(vi_tok_path))
else:
    print('⏳ Đang huấn luyện BPE Tokenizer tiếng Anh mới...')
    tok_en = TranslationTokenizer()
    tok_en.train(train_en, vocab_size=VOCAB_SIZE)
    tok_en.tokenizer.save(en_tok_path)

    print('⏳ Đang huấn luyện BPE Tokenizer tiếng Việt mới...')
    tok_vi = TranslationTokenizer()
    tok_vi.train(train_vi, vocab_size=VOCAB_SIZE)
    tok_vi.tokenizer.save(vi_tok_path)

SRC_VOCAB_SIZE = tok_en.tokenizer.get_vocab_size()
TGT_VOCAB_SIZE = tok_vi.tokenizer.get_vocab_size()
print(f'✅ Tokenizer sẵn sàng! Vocab EN: {SRC_VOCAB_SIZE:,}, Vocab VI: {TGT_VOCAB_SIZE:,}')


In [ ]:
# 4. PyTorch Dataset & Dynamic Padding Masking
def create_padding_mask(seq, pad_idx=PAD_IDX):
    return (seq != pad_idx).unsqueeze(1).unsqueeze(2)

def create_causal_mask(seq_len, device=None):
    mask = torch.tril(torch.ones((seq_len, seq_len), device=device)).bool()
    return mask.unsqueeze(0).unsqueeze(1)

def create_tgt_mask(tgt, pad_idx=PAD_IDX):
    batch_size, tgt_len = tgt.size()
    pad_mask = create_padding_mask(tgt, pad_idx)
    causal_mask = create_causal_mask(tgt_len, tgt.device)
    return pad_mask & causal_mask

class TranslationDataset(Dataset):
    def __init__(self, en_texts, vi_texts, tok_en, tok_vi, max_len=80):
        self.pairs = []
        for en, vi in zip(en_texts, vi_texts):
            en_ids = tok_en.encode(en, add_special_tokens=True)
            vi_ids = tok_vi.encode(vi, add_special_tokens=True)
            if len(en_ids) <= max_len and len(vi_ids) <= max_len:
                self.pairs.append((en_ids, vi_ids))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]

def collate_fn(batch):
    src_list, tgt_list = zip(*batch)
    max_src = max(len(s) for s in src_list)
    max_tgt = max(len(t) for t in tgt_list)
    batch_size = len(batch)

    src_pad = torch.full((batch_size, max_src), PAD_IDX, dtype=torch.long)
    tgt_pad = torch.full((batch_size, max_tgt), PAD_IDX, dtype=torch.long)

    for i, (s, t) in enumerate(zip(src_list, tgt_list)):
        src_pad[i, :len(s)] = torch.tensor(s, dtype=torch.long)
        tgt_pad[i, :len(t)] = torch.tensor(t, dtype=torch.long)

    tgt_input = tgt_pad[:, :-1]
    tgt_label = tgt_pad[:, 1:]
    src_mask = create_padding_mask(src_pad)
    tgt_mask = create_tgt_mask(tgt_input)
    return src_pad, tgt_input, tgt_label, src_mask, tgt_mask

train_dataset = TranslationDataset(train_en, train_vi, tok_en, tok_vi)
dev_dataset = TranslationDataset(dev_en, dev_vi, tok_en, tok_vi)

# BATCH_SIZE = 128 tối ưu cho L4 (hoặc 96 cho T4)
BATCH_SIZE = 128 if 'L4' in torch.cuda.get_device_name(0) else 96 if torch.cuda.is_available() else 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f'✅ DataLoaders sẵn sàng! Batch Size = {BATCH_SIZE} | Số batches Train: {len(train_loader):,}')


In [ ]:
# 5. Kiến trúc Transformer Seq2Seq (PyTorch from Scratch)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        bs = q.size(0)
        q = self.w_q(q).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        k = self.w_k(k).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)
        v = self.w_v(v).view(bs, -1, self.n_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e4)  # Dùng -1e4 an toàn tuyệt đối với FP16 Half
        attn = self.dropout(F.softmax(scores, dim=-1))
        output = torch.matmul(attn, v).transpose(1, 2).contiguous().view(bs, -1, self.d_model)
        return self.w_o(output)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()

    def forward(self, x):
        return self.fc2(self.dropout(self.act(self.fc1(x))))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = x + self.dropout(self.attn(self.norm1(x), self.norm1(x), self.norm1(x), mask))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        x = x + self.dropout(self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), tgt_mask))
        x = x + self.dropout(self.cross_attn(self.norm2(x), enc_out, enc_out, src_mask))
        x = x + self.dropout(self.ffn(self.norm3(x)))
        return x

class TransformerSeq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, n_heads=8, d_ff=2048, num_layers=4, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.src_emb = nn.Embedding(src_vocab_size, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)

        self.enc_layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)])
        self.dec_layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)])

        self.enc_norm = nn.LayerNorm(d_model)
        self.dec_norm = nn.LayerNorm(d_model)
        self.out_linear = nn.Linear(d_model, tgt_vocab_size, bias=False)

        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)

    def encode(self, src, src_mask=None):
        x = self.pos_enc(self.src_emb(src) * math.sqrt(self.d_model))
        for layer in self.enc_layers:
            x = layer(x, src_mask)
        return self.enc_norm(x)

    def decode(self, tgt, enc_out, src_mask=None, tgt_mask=None):
        x = self.pos_enc(self.tgt_emb(tgt) * math.sqrt(self.d_model))
        for layer in self.dec_layers:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return self.dec_norm(x)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_out = self.encode(src, src_mask)
        dec_out = self.decode(tgt, enc_out, src_mask, tgt_mask)
        return self.out_linear(dec_out)

model = TransformerSeq2Seq(
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,
    d_model=512, n_heads=8, d_ff=2048, num_layers=4, dropout=0.1
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Mô hình Transformer khởi tạo thành công! Tổng số tham số: {total_params / 1e6:.2f}M')


In [ ]:
# 6. Vòng lặp huấn luyện nâng cao (Hỗ trợ Resume Training từ Checkpoint cũ)
EPOCHS = 8
LR = 0.0003  # Giảm nhẹ LR để fine-tune sâu
WARMUP_STEPS = 2000

# Kiểm tra và nạp Checkpoint cũ nếu có để train tiếp (Epoch 9 -> 16)
CKPT_PATH = os.path.join(SAVE_DIR, 'best_transformer.pt')
start_epoch = 1
best_val_loss = float('inf')

if os.path.exists(CKPT_PATH):
    try:
        ckpt = torch.load(CKPT_PATH, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        start_epoch = ckpt.get('epoch', 0) + 1
        best_val_loss = ckpt.get('val_loss', float('inf'))
        print(f'⭐ ĐÃ NẠP CHECKPOINT THÀNH CÔNG! Huấn luyện tiếp từ Epoch {start_epoch} (Val Loss cũ: {best_val_loss:.4f})')
    except Exception as e:
        print(f'⚠️ Không thể nạp checkpoint ({e}). Bắt đầu huấn luyện mới.')
else:
    print('🚀 Bắt đầu huấn luyện mới từ Epoch 1.')

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.98), eps=1e-9, weight_decay=1e-4)

# Noam Scheduler
def get_lr(step, d_model=512, warmup=WARMUP_STEPS):
    step = max(1, step)
    return (d_model ** -0.5) * min(step ** -0.5, step * (warmup ** -1.5))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda s: get_lr(s) / get_lr(WARMUP_STEPS))
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

history = {'train_loss': [], 'val_loss': []}
total_epochs = start_epoch + EPOCHS - 1

print(f'🚀 Bắt đầu vòng lặp huấn luyện từ Epoch {start_epoch} đến Epoch {total_epochs}...')
for epoch in range(start_epoch, total_epochs + 1):
    start_time = time.time()
    model.train()
    train_loss = 0.0
    total_batches = len(train_loader)

    for step, (src, tgt_in, tgt_lbl, s_mask, t_mask) in enumerate(train_loader, 1):
        src, tgt_in, tgt_lbl = src.to(device), tgt_in.to(device), tgt_lbl.to(device)
        s_mask, t_mask = s_mask.to(device), t_mask.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(src, tgt_in, s_mask, t_mask)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_lbl.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()

        if step % 500 == 0 or step == total_batches:
            cur_lr = optimizer.param_groups[0]['lr']
            print(f'   Epoch [{epoch}/{total_epochs}] - Batch [{step}/{total_batches}] | Loss: {loss.item():.4f} | LR: {cur_lr:.6f}')

    avg_train_loss = train_loss / total_batches

    # Đánh giá trên tập Dev
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for src, tgt_in, tgt_lbl, s_mask, t_mask in dev_loader:
            src, tgt_in, tgt_lbl = src.to(device), tgt_in.to(device), tgt_lbl.to(device)
            s_mask, t_mask = s_mask.to(device), t_mask.to(device)
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                logits = model(src, tgt_in, s_mask, t_mask)
                loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_lbl.reshape(-1))
            val_loss += loss.item()
    avg_val_loss = val_loss / len(dev_loader)
    elapsed = time.time() - start_time

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)

    print(f'⭐ HOÀN TẤT Epoch [{epoch:02d}/{total_epochs:02d}] | Thời gian: {elapsed/60:.1f} phút | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')

    # Lưu checkpoint tốt nhất
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'config': {'d_model': 512, 'n_heads': 8, 'd_ff': 2048, 'num_layers': 4, 'dropout': 0.1},
            'src_vocab_size': SRC_VOCAB_SIZE,
            'tgt_vocab_size': TGT_VOCAB_SIZE,
            'val_loss': best_val_loss,
            'epoch': epoch
        }, CKPT_PATH)
        print(f'   💾 Đã cập nhật Model tốt nhất vào Google Drive: {CKPT_PATH}')


In [ ]:
# 7. Thử nghiệm dịch thuật với Beam Search + Chặn lặp từ (N-gram Blocking)
@torch.no_grad()
def translate(text, beam_size=4, max_len=80, no_repeat_ngram_size=3):
    model.eval()
    src_ids = tok_en.encode(text, add_special_tokens=True)
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    src_mask = create_padding_mask(src)
    enc_out = model.encode(src, src_mask)

    beams = [([SOS_IDX], 0.0)]
    for _ in range(max_len):
        candidates = []
        for tokens, score in beams:
            if tokens[-1] == EOS_IDX:
                candidates.append((tokens, score))
                continue
            tgt = torch.tensor([tokens], dtype=torch.long, device=device)
            tgt_mask = create_tgt_mask(tgt)
            logits = model.out_linear(model.decode(tgt, enc_out, src_mask, tgt_mask))
            log_probs = F.log_softmax(logits[:, -1, :], dim=-1).squeeze(0)

            # Chặn lặp n-gram (no-repeat n-gram blocking)
            if no_repeat_ngram_size > 0 and len(tokens) >= no_repeat_ngram_size:
                ngram_prefix = tuple(tokens[-(no_repeat_ngram_size - 1):])
                for i in range(len(tokens) - no_repeat_ngram_size + 1):
                    if tuple(tokens[i:i + no_repeat_ngram_size - 1]) == ngram_prefix:
                        banned_token = tokens[i + no_repeat_ngram_size - 1]
                        log_probs[banned_token] = -1e4

            topk_probs, topk_ids = torch.topk(log_probs, beam_size)
            for k in range(beam_size):
                candidates.append((tokens + [topk_ids[k].item()], score + topk_probs[k].item()))
        candidates.sort(key=lambda x: x[1], reverse=True)
        beams = candidates[:beam_size]
        if all(tokens[-1] == EOS_IDX for tokens, _ in beams):
            break

    best_tokens, _ = max(beams, key=lambda x: x[1] / (((5.0 + len(x[0])) / 6.0) ** 0.7))
    return tok_vi.decode(best_tokens)

# Kiểm tra các câu từ vựng chuyên ngành và câu đối thoại thực tế
test_sentences = [
    'Hello, how are you today?',
    'Artificial intelligence and machine learning are the future of software engineering.',
    'This is a smartphone with high resolution camera.',
    'Inflation rate has a significant impact on global economy.',
    'Deep neural network architectures require massive parallel computation.'
]

print('\n===== KẾT QUẢ DỊCH THỰC TẾ (BEAM SEARCH + N-GRAM BLOCKING) =====')
for s in test_sentences:
    translated = translate(s, beam_size=4)
    print(f'EN: {s}')
    print(f'VI: {translated}\n')


In [ ]:
# 8. Vẽ biểu đồ học tập & Thông tin Checkpoint
import matplotlib.pyplot as plt

if history['train_loss']:
    plt.figure(figsize=(8, 5))
    epochs_range = range(start_epoch, start_epoch + len(history['train_loss']))
    plt.plot(epochs_range, history['train_loss'], label='Train Loss', marker='o')
    plt.plot(epochs_range, history['val_loss'], label='Validation Loss', marker='s')
    plt.xlabel('Epoch')
    plt.ylabel('CrossEntropy Loss')
    plt.title('Đường cong học tập (Learning Curve)')
    plt.legend()
    plt.grid(True)
    plt.show()

print(f'🎉 CHECKPOINT ĐÃ SẴN SÀNG TẠI: {CKPT_PATH}')
print('Tải 3 file: best_transformer.pt, tokenizer_en.json, tokenizer_vi.json về thư mục Translator/models/ để chạy Web UI!')
